# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [1]:
%uv pip install -q torch transformers accelerate huggingface_hub hf_transfer safetensors

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import re
from pathlib import Path
from collections import Counter

import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
# Use this only if the merged repo is private.
# Requires HF_TOKEN in the environment.
login(token="")

In [23]:
MODEL_ID = "isji/sinllama-1b-qa-merged"

# Works in Modal whether you upload test.jsonl to /tmp or /tmp/splits.
TEST_FILE_CANDIDATES = [
    Path("/tmp/test.jsonl")
]
TEST_FILE = next((path for path in TEST_FILE_CANDIDATES if path.exists()), TEST_FILE_CANDIDATES[0])
OUTPUT_FILE = Path("test_predictions.jsonl")
MAX_NEW_TOKENS = 80

NO_ANSWER = "මෙම ප්‍රශ්නයට පිළිතුරු දීමට ප්‍රමාණවත් තොරතුරු නොමැත."

print("Using test file:", TEST_FILE.resolve())
print("Exists:", TEST_FILE.exists())

Using test file: /tmp/test.jsonl
Exists: True


In [24]:
print(f"Loading merged model: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

print("Merged QA model loaded.")

Loading merged model: isji/sinllama-1b-qa-merged


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/19.3M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/833 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

  2026-07-27T09:46:48.883811Z  WARN  Status Code: 500. Retrying..., request_id: "01KYHFFW5HYSN33TCJZHFVYB2J"
    at /home/runner/work/xet-core/xet-core/cas_client/src/http_client.rs:220

  2026-07-27T09:46:48.884001Z  WARN  Retry attempt #0. Sleeping 644.082397ms before the next attempt
    at /root/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/reqwest-retry-0.7.0/src/middleware.rs:171



generation_config.json:   0%|          | 0.00/180 [00:00<?, ?B/s]

Merged QA model loaded.


In [26]:
def build_qa_prompt(context, question):
    return (
        "උපදෙස්: ඔබ සිංහල ප්‍රශ්න-පිළිතුරු සහායකයෙකි. පහත සපයා ඇති තොරතුරු (Context) පමණක් භාවිතා කරමින් "
        "ප්‍රශ්නයට නිවැරදි, සෘජු, කෙටි පිළිතුරක් දෙන්න. Context තුළ පිළිතුර නොමැති නම් හෝ ප්‍රමාණවත් තොරතුරු නොමැති නම්, "
        f"“{NO_ANSWER}” යනුවෙන් පමණක් පිළිතුරු දෙන්න. Context වලින් පිටත දැනුම, අනුමාන, හෝ අමතර විස්තර භාවිතා නොකරන්න.\n\n"
        f"තොරතුරු (Context): {context}\n"
        f"ප්‍රශ්නය: {question}\n"
        "පිළිතුර: ["
    )

def run_qa(context, question, max_new_tokens=MAX_NEW_TOKENS):
    prompt = build_qa_prompt(context, question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = full_text.split("පිළිතුර: [")[-1].strip()
    if "]" in answer:
        answer = answer.split("]", 1)[0].strip()
    return answer

print("Inference helper ready.")

Inference helper ready.


In [27]:
def normalize_answer(text):
    text = (text or "").strip()
    text = text.replace("]", "").replace("[", "")
    text = re.sub(r"\s+", " ", text)
    for suffix in (" ය.", "යි.", " ය", "යි"):
        if text.endswith(suffix):
            text = text[: -len(suffix)].strip()
    return text

def keyword_match(prediction, gold):
    pred = normalize_answer(prediction)
    target = normalize_answer(gold)
    if not pred or not target:
        return False
    if pred == target:
        return True
    if target in pred or pred in target:
        return True

    target_tokens = [token for token in target.split() if len(token) > 1]
    if not target_tokens:
        return False
    matched = sum(1 for token in target_tokens if token in pred)
    return matched / len(target_tokens) >= 0.6

print("Scoring helpers ready.")

Scoring helpers ready.


In [28]:
test_rows = []
with TEST_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            test_rows.append(json.loads(line))

print(f"Loaded test rows: {len(test_rows)}")
print(Counter(row["answerable"] for row in test_rows))

Loaded test rows: 153
Counter({True: 135, False: 18})


In [29]:
# Optional quick smoke test before running all 194 rows.
row = test_rows[0]
pred = run_qa(row["context"], row["question"])

print("Question:", row["question"])
print("Gold:", NO_ANSWER if row["answerable"] is False else row["answer"])
print("Prediction:", pred)

Question: 1818 නොවැම්බර් 26 දින ඉංග්‍රීසීන් විසින් හිස ගසා මරණයට පත් කරන ලද සටනේ නායකයන් දෙදෙනා කවුද?
Gold: කැප්පෙටිපොළ හා මඩුගල්ලේ ය.
Prediction: කැප්පෙටිපොළ හා මඩුගල්ලේ ය.


In [30]:
predictions = []
exact = 0
soft = 0
no_answer_correct = 0
unanswerable_total = 0

for idx, row in enumerate(test_rows, start=1):
    gold = NO_ANSWER if row["answerable"] is False else row["answer"]
    pred = run_qa(row["context"], row["question"])

    exact_ok = normalize_answer(pred) == normalize_answer(gold)
    soft_ok = keyword_match(pred, gold)
    no_answer_ok = row["answerable"] is False and normalize_answer(pred) == normalize_answer(NO_ANSWER)

    exact += int(exact_ok)
    soft += int(soft_ok)
    if row["answerable"] is False:
        unanswerable_total += 1
        no_answer_correct += int(no_answer_ok)

    predictions.append({
        "index": idx,
        "grade": row.get("grade"),
        "chapter": row.get("chapter"),
        "answerable": row.get("answerable"),
        "context": row.get("context"),
        "question": row.get("question"),
        "gold_answer": gold,
        "prediction": pred,
        "exact_match": exact_ok,
        "soft_match": soft_ok,
    })

    if idx % 10 == 0 or idx == len(test_rows):
        print(f"Done {idx}/{len(test_rows)}")

print("Finished inference.")

Done 10/153
Done 20/153
Done 30/153
Done 40/153
Done 50/153
Done 60/153
Done 70/153
Done 80/153
Done 90/153
Done 100/153
Done 110/153
Done 120/153
Done 130/153
Done 140/153
Done 150/153
Done 153/153
Finished inference.


In [18]:
total = len(predictions)
print("=== Test Results ===")
print(f"Total: {total}")
print(f"Exact match: {exact}/{total} ({(exact / max(total, 1)) * 100:.1f}%)")
print(f"Soft match: {soft}/{total} ({(soft / max(total, 1)) * 100:.1f}%)")
print(
    f"No-answer exact: {no_answer_correct}/{unanswerable_total} "
    f"({(no_answer_correct / max(unanswerable_total, 1)) * 100:.1f}%)"
)

=== Test Results ===
Total: 153
Exact match: 11/153 (7.2%)
Soft match: 37/153 (24.2%)
No-answer exact: 0/18 (0.0%)


In [31]:
with OUTPUT_FILE.open("w", encoding="utf-8", newline="\n") as f:
    for item in predictions:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved predictions to {OUTPUT_FILE}")

Saved predictions to test_predictions.jsonl


In [32]:
# Inspect failed examples.
failed = [item for item in predictions if not item["soft_match"]]
print(f"Soft-match failures: {len(failed)}")

for item in failed[:10]:
    print("-" * 80)
    print("Q:", item["question"])
    print("Gold:", item["gold_answer"])
    print("Pred:", item["prediction"])

Soft-match failures: 85
--------------------------------------------------------------------------------
Q: මුල්ලේරියා සටනේදී සීතාවක හමුදාව මෙහෙයවූයේ කවුද?
Gold: ටිකිරි බණ්ඩාර හෙවත් පළමුවන රාජසිංහ රජු විසිනි.
Pred: ඔහුගේ පුත් ටිකිරි බණ්ඩාර ය.
--------------------------------------------------------------------------------
Q: 1818 සටන ආරම්භයේදී සටන්කාමීන් අතින් ඝාතනය වූ බදුල්ලේ ඒජන්තවරයා කවුද?
Gold: ඩග්ලස් විල්සන් ය.
Pred: T. විල්සන් ය.
--------------------------------------------------------------------------------
Q: 1848 සටනේදී පුරන් අප්පු විසින් භාවිතා කළ තුවක්කු වර්ගය කුමක්ද?
Gold: මෙම ප්‍රශ්නයට පිළිතුරු දීමට ප්‍රමාණවත් තොරතුරු නොමැත.
Pred: සපයා ඇති සන්දර්භය මත පදනම්ව මෙය පිළිතුරු දිය නොහැක.
--------------------------------------------------------------------------------
Q: මිනිපේ ඇළ තනවන ලද්දේ කුමන රජතුමා විසින්ද?
Gold: පළමුවන අග්ගබෝධි රජු විසිනි.
Pred: මහසෙන් රජතුමා විසිනි.
--------------------------------------------------------------------------------
Q: කෝට්ටේ යුගයේ අධ්‍යාපන ප

In [33]:
# Print generated and expected answers for all test QAs.
for item in predictions:
    print("-" * 100)
    print(f"Index     : {item['index']}")
    print(f"Grade     : {item.get('grade')} | Chapter: {item.get('chapter')}")
    print(f"Answerable: {item.get('answerable')}")
    print(f"Question  : {item['question']}")
    print(f"Expected  : {item['gold_answer']}")
    print(f"Generated : {item['prediction']}")
    print(f"Exact     : {item['exact_match']} | Soft: {item['soft_match']}")

----------------------------------------------------------------------------------------------------
Index     : 1
Grade     : 11 | Chapter: 2
Answerable: True
Question  : 1818 නොවැම්බර් 26 දින ඉංග්‍රීසීන් විසින් හිස ගසා මරණයට පත් කරන ලද සටනේ නායකයන් දෙදෙනා කවුද?
Expected  : කැප්පෙටිපොළ හා මඩුගල්ලේ ය.
Generated : කැප්පෙටිපොළ හා මඩුගල්ලේ ය.
Exact     : True | Soft: True
----------------------------------------------------------------------------------------------------
Index     : 2
Grade     : 9 | Chapter: 4
Answerable: True
Question  : ඉංග්‍රීසීන් යටතේ ඉන්දියාවේ හමුදා සේවයට බඳවාගෙන සිටි ඉන්දියානුවන් හැඳින්වූයේ කෙසේද?
Expected  : සිපොයි හමුදාව වශයෙනි.
Generated : සිපොයි හමුදාව වශයෙනි.
Exact     : True | Soft: True
----------------------------------------------------------------------------------------------------
Index     : 3
Grade     : 8 | Chapter: 5
Answerable: True
Question  : මුල්ලේරියා සටනේදී සීතාවක හමුදාව මෙහෙයවූයේ කවුද?
Expected  : ටිකිරි බණ්ඩාර හෙවත් පළමුවන රාජසිංහ රජු විසිනි